### Para el análisis

Generación de tablas por Hoja

In [2]:
import tkinter as tk
from tkinter import filedialog, messagebox
import openpyxl
from openpyxl.utils import get_column_letter
from openpyxl.styles import PatternFill
import os
import unicodedata
import re
import math
from collections import Counter
import difflib
import pandas as pd


def normalizar_localidad(texto):
    if pd.isna(texto): return "SIN REGISTRO"
    texto = str(texto).strip().upper()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    # Versión sin espacios ni signos: así "SANTA FE", "Santa fe", "SANTAFE", "Santafé",
    # "Santa-Fe", "SAN CRISTOBAL"/"SANCRISTOBAL", etc. se reconocen igual.
    t = re.sub(r'[^A-Z]', '', texto)

    if "USAQUEN" in t: return "01 - USAQUÉN"
    if "CHAPINERO" in t: return "02 - CHAPINERO"
    if "SANTAFE" in t: return "03 - SANTA FE"
    if "SANCRISTOBAL" in t: return "04 - SAN CRISTÓBAL"
    if "USME" in t: return "05 - USME"
    if "TUNJUELITO" in t: return "06 - TUNJUELITO"
    if "BOSA" in t: return "07 - BOSA"
    if "KENNEDY" in t: return "08 - KENNEDY"
    if "FONTIBON" in t: return "09 - FONTIBÓN"
    if "ENGATIVA" in t: return "10 - ENGATIVÁ"
    if "BARRIOSUNIDOS" in t: return "12 - BARRIOS UNIDOS"
    if "SUBA" in t: return "11 - SUBA"
    if "TEUSAQUILLO" in t: return "13 - TEUSAQUILLO"
    if "MARTIRES" in t: return "14 - LOS MÁRTIRES"
    if "ANTONIONARINO" in t: return "15 - ANTONIO NARIÑO"
    if "PUENTEARANDA" in t: return "16 - PUENTE ARANDA"
    if "CANDELARIA" in t: return "17 - LA CANDELARIA"
    if "RAFAELURIBE" in t: return "18 - RAFAEL URIBE URIBE"
    if "CIUDADBOLIVAR" in t: return "19 - CIUDAD BOLÍVAR"
    if "SUMAPAZ" in t: return "20 - SUMAPAZ"

    return texto.replace("ALCALDIA LOCAL DE ", "").strip()


ORDEN_LOCALIDADES = [
    "01 - USAQUÉN", "02 - CHAPINERO", "03 - SANTA FE", "04 - SAN CRISTÓBAL",
    "05 - USME", "06 - TUNJUELITO", "07 - BOSA", "08 - KENNEDY",
    "09 - FONTIBÓN", "10 - ENGATIVÁ", "11 - SUBA", "12 - BARRIOS UNIDOS",
    "13 - TEUSAQUILLO", "14 - LOS MÁRTIRES", "15 - ANTONIO NARIÑO",
    "16 - PUENTE ARANDA", "17 - LA CANDELARIA", "18 - RAFAEL URIBE URIBE",
    "19 - CIUDAD BOLÍVAR", "20 - SUMAPAZ",
]


def ordenar_por_localidad(tabla, filas_finales=("Total", "PROMEDIO GENERAL")):
    """Ordena las filas 01..20, luego localidades raras, y al final Total/Promedio general."""
    finales = [f for f in filas_finales if f in tabla.index]
    oficiales = [l for l in ORDEN_LOCALIDADES if l in tabla.index]
    otras = [l for l in tabla.index if l not in oficiales and l not in finales]
    return tabla.reindex(oficiales + otras + finales)

# ==========================================
# CLASE NLP: MOTOR DE RECUPERACIÓN RANQUEADA
# (Se mantiene exactamente igual, sin cambios)
# ==========================================
class MotorRecuperacion:
    def __init__(self, stop_words=None, n_max=3): 
        self.stop_words = stop_words if stop_words else set()
        self.n_max = n_max 
        self.idf = {}
        self.tf_idf_enciclopedia = [] 
        self.magnitudes_enciclopedia = []
        self.etiquetas_categorias = []
        self.vocabulario_conocido = set() 

    def normalizar(self, texto):
        if not texto: return ""
        texto = ''.join(c for c in unicodedata.normalize('NFD', str(texto)) if unicodedata.category(c) != 'Mn')
        texto = texto.lower()
        texto = re.sub(r'[^a-z0-9\s]', '', texto)
        return texto

    def corregir_ortografia(self, texto):
        palabras = self.normalizar(texto).split()
        palabras_corregidas = []
        for p in palabras:
            if p in self.stop_words:
                continue
            if p in self.vocabulario_conocido:
                palabras_corregidas.append(p)
            else:
                matches = difflib.get_close_matches(p, list(self.vocabulario_conocido), n=1, cutoff=0.75)
                if matches:
                    palabras_corregidas.append(matches[0]) 
                else:
                    palabras_corregidas.append(p) 
        return " ".join(palabras_corregidas)

    def obtener_ngramas_combinados(self, texto):
        palabras = texto.split() 
        if not palabras: return []
        ngramas = []
        rango_maximo = min(self.n_max, len(palabras))
        for n in range(1, rango_maximo + 1):
            for i in range(len(palabras) - n + 1):
                ngramas.append(" ".join(palabras[i:i+n]))
        return ngramas

    def cargar_enciclopedia(self, nombres_categorias, documentos_entrenamiento):
        self.etiquetas_categorias = nombres_categorias
        conteo_docs_con_termino = Counter()
        documentos_tokenizados = []
        
        for doc in documentos_entrenamiento:
            palabras = self.normalizar(doc).split()
            for p in palabras:
                if p not in self.stop_words:
                    self.vocabulario_conocido.add(p)
        
        for doc in documentos_entrenamiento:
            texto_limpio = " ".join([p for p in self.normalizar(doc).split() if p not in self.stop_words])
            tokens = self.obtener_ngramas_combinados(texto_limpio)
            documentos_tokenizados.append(tokens)
            for token in set(tokens):
                conteo_docs_con_termino[token] += 1
        
        N = len(documentos_entrenamiento)
        for token, freq in conteo_docs_con_termino.items():
            self.idf[token] = math.log10((1.0 + N) / freq)

        for tokens in documentos_tokenizados:
            tf_vector = Counter(tokens)
            vector_tfidf = {}
            suma_cuadrados = 0
            for token, count in tf_vector.items():
                tf_w = 1.0 + math.log10(count)
                val_tfidf = tf_w * self.idf[token]
                vector_tfidf[token] = val_tfidf
                suma_cuadrados += val_tfidf ** 2
            self.tf_idf_enciclopedia.append(vector_tfidf)
            self.magnitudes_enciclopedia.append(math.sqrt(suma_cuadrados))

    def clasificar_query(self, texto_query, umbral_similitud=0.025):
        texto_corregido = self.corregir_ortografia(texto_query)
        tokens_query = self.obtener_ngramas_combinados(texto_corregido)
        
        if not tokens_query: 
            return "-", 0.0, "Sin palabras válidas"

        tf_query = Counter(tokens_query)
        vector_query = {}
        suma_cuadrados_query = 0
        for token, count in tf_query.items():
            if token in self.idf:
                tf_w = 1.0 + math.log10(count)
                val_tfidf = tf_w * self.idf[token]
                vector_query[token] = val_tfidf
                suma_cuadrados_query += val_tfidf ** 2

        mag_query = math.sqrt(suma_cuadrados_query)
        if mag_query == 0: 
            return "-", 0.0, "Coseno = 0"

        resultados = []
        for idx, vec_cat in enumerate(self.tf_idf_enciclopedia):
            producto_punto = sum(vector_query.get(t, 0) * vec_cat.get(t, 0) for t in vec_cat)
            mag_cat = self.magnitudes_enciclopedia[idx]
            similitud = producto_punto / (mag_query * mag_cat) if mag_cat > 0 else 0
            resultados.append((self.etiquetas_categorias[idx], similitud))

        resultados.sort(key=lambda x: x[1], reverse=True)
        mejor_categoria = resultados[0][0]
        mejor_score = resultados[0][1]

        debug_str = " | ".join([f"{cat}: {score:.3f}" for cat, score in resultados if score > 0])
        if not debug_str:
            debug_str = "0.0 en todas"

        if mejor_score >= umbral_similitud:
            return mejor_categoria, mejor_score, debug_str
        return "-", mejor_score, debug_str

# ==========================================
# INTERFAZ Y PROCESAMIENTO EXCEL (RESTAURADO)
# ==========================================
def normalizar_texto(texto):
    if texto is None:
        return ""
    texto = str(texto).strip().upper()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'[\s_\-]+', '', texto)
    return texto


# ==========================================
# NUEVO: UTILIDADES PARA LAS HOJAS ANALÍTICAS
# ==========================================
# Si True, los casos devueltos a la DGDL salen de su estado original (p. ej. CERRADO)
# y se cuentan SOLO en "CERRADO devuelto a la DGDL" (así el Total no cuenta doble).
# Si False, se siguen contando en su estado original y además en la fila nueva.
SEPARAR_DEVUELTOS_DGDL = True
ESTADO_DGDL = "CERRADO devuelto a la DGDL"
TEXTO_DGDL = "DIRECCION PARA LA GESTION"   # se compara sin tildes y en mayúsculas


def buscar_columna_df(df, *nombres):
    """Busca una columna del DataFrame ignorando tildes, espacios, guiones y mayúsculas."""
    objetivos = {normalizar_texto(n) for n in nombres}
    for c in df.columns:
        if normalizar_texto(c) in objetivos:
            return c
    return None


def texto_sin_tildes(serie):
    """Pasa una serie a texto en mayúsculas, sin tildes y con espacios simples."""
    return (serie.fillna("").astype(str)
            .map(lambda t: ''.join(ch for ch in unicodedata.normalize('NFD', t)
                                   if unicodedata.category(ch) != 'Mn'))
            .str.upper()
            .str.replace(r'\s+', ' ', regex=True))


def a_numero(serie):
    """Convierte a número; si viene como texto ('120 min', '1,5') extrae el valor."""
    num = pd.to_numeric(serie, errors='coerce')
    faltan = num.isna() & serie.notna()
    if faltan.any():
        extra = (serie[faltan].astype(str).str.replace(',', '.', regex=False)
                 .str.extract(r'(-?\d+(?:\.\d+)?)')[0].astype(float))
        num.loc[faltan] = extra
    return num


def procesar_excel():
    root = tk.Tk()
    root.withdraw()
    
    print("Paso 1: Seleccionando archivo TXT de categorías...")
    ruta_txt = filedialog.askopenfilename(
        title="Selecciona archivo de Categorías (.txt)",
        filetypes=[("Archivos de Texto", "*.txt")]
    )
    if not ruta_txt: return

    print("Paso 1.1: Seleccionando archivo TXT de Clústeres...")
    ruta_cluster = filedialog.askopenfilename(
        title="Selecciona archivo de Clústeres (.txt)", 
        filetypes=[("Archivos de Texto", "*.txt")]
    )
    dict_clusters = {}
    if ruta_cluster:
        with open(ruta_cluster, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    # Intenta separar por tabulador, pipe, punto y coma o coma
                    for sep in ['\t', '|', ';', ',']:
                        if sep in line:
                            partes = line.split(sep)
                            dict_clusters[partes[0].strip()] = partes[1].strip()
                            break
    
    print("Paso 1.2: Seleccionando Excel de mapeo de Clústeres 2...")
    ruta_cluster2 = filedialog.askopenfilename(title="Archivo Clústeres 2 (.xlsx)", filetypes=[("Archivos de Texto", "*.txt")])
    dict_clusters_2 = {}
    if ruta_cluster2:
            with open(ruta_cluster2, 'r', encoding='utf-8') as f:
                for line in f:
                    if line.strip():
                        # Intenta separar por tabulador, pipe, punto y coma o coma
                        for sep in ['\t', '|', ';', ',']:
                            if sep in line:
                                partes = line.split(sep)
                                dict_clusters_2[partes[0].strip()] = partes[1].strip()
                                break

    print("Paso 1.3: Seleccionando archivo TXT con lista de conteo (Tercera Hoja)...")
    ruta_txt_lista = filedialog.askopenfilename(
        title="Selecciona archivo Lista Específica (.txt)", 
        filetypes=[("Archivos de Texto", "*.txt")]
    )
    lista_conteo_personalizada = []
    if ruta_txt_lista:
        with open(ruta_txt_lista, 'r', encoding='utf-8') as f:
            lista_conteo_personalizada = [line.strip().upper() for line in f if line.strip()]

    nombres_categorias = []
    documentos_entrenamiento = []
    
    try:
        with open(ruta_txt, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    if '|' in line:
                        partes = line.split('|')
                        etiqueta = partes[0].strip()
                        sinonimos = partes[1].strip()
                        nombres_categorias.append(etiqueta)
                        documentos_entrenamiento.append(etiqueta + " " + sinonimos)
                    else:
                        nombres_categorias.append(line.strip())
                        documentos_entrenamiento.append(line.strip())
    except Exception as e:
        print(f"Error al leer TXT: {e}")
        return

    # Entrenar el motor
    stop_words_espanol = {'de', 'la', 'el', 'en', 'y', 'que', 'los', 'las', 'un', 'una', 'al', 'por', 'con', 'para'}
    try:
        with open("Stopwords-es.txt", "r", encoding="utf-8") as archivo:
            stop_words_espanol = set([linea.strip() for linea in archivo if linea.strip()])
    except:
        print("Aviso: No se encontró 'Stopwords-es.txt', usando stop words por defecto.")
        
    motor = MotorRecuperacion(stop_words=stop_words_espanol, n_max=3)
    motor.cargar_enciclopedia(nombres_categorias, documentos_entrenamiento)

    print("Paso 2: Seleccionando archivo Excel a procesar...")
    ruta_archivo = filedialog.askopenfilename(
        title="Selecciona el archivo Excel",
        filetypes=[("Archivos Excel", "*.xlsx")]
    )
    if not ruta_archivo: return

    try:
        print("Procesando Excel...")
        wb = openpyxl.load_workbook(ruta_archivo)
        ws = wb.active
        max_row = ws.max_row
        
        # Color amarillo para las nuevas columnas
        color_fondo = PatternFill(start_color="FFF2CC", end_color="FFF2CC", fill_type="solid")
        
        # 1. DETECTAR EN QUÉ FILA ESTÁN LOS ENCABEZADOS
        fila_cabecera = 1
        encontrado = False
        for r in range(1, 16):
            for c in range(1, ws.max_column + 1):
                val = normalizar_texto(ws.cell(row=r, column=c).value)
                if val in ["DESCRIPCION", "FECHACIERRE", "TIEMPODESOLUCIONREAL", "ULTIMOCAMBIOASOCIADO"]:
                    fila_cabecera = r
                    encontrado = True
                    break
            if encontrado:
                break
                
        def buscar_columna(nombre_objetivo):
            nombre_limpio = normalizar_texto(nombre_objetivo)
            for col in range(1, ws.max_column + 1):
                valor_celda = ws.cell(row=fila_cabecera, column=col).value
                if normalizar_texto(valor_celda) == nombre_limpio:
                    return col
            return None

        columnas_no_encontradas = []

        # ==========================================
        # FASE 1: CREAR ESTRUCTURA E INSERTAR COLUMNAS
        # ==========================================
        
        # 1. ULTIMO CAMBIO ASOCIADO
        col = buscar_columna("ULTIMOCAMBIOASOCIADO")
        if col:
            ws.insert_cols(col + 1)
            ws.cell(row=fila_cabecera, column=col + 1, value="Solución").fill = color_fondo
        else:
            columnas_no_encontradas.append("ULTIMO CAMBIO ASOCIADO")

        # 2. TIEMPO DE SOLUCION REAL
        col = buscar_columna("TIEMPODESOLUCIONREAL")
        if col:
            ws.insert_cols(col + 1, amount=3)
            ws.cell(row=fila_cabecera, column=col + 1, value="Tiempos total de atención").fill = color_fondo
            ws.cell(row=fila_cabecera, column=col + 2, value="formula tiempos por hora").fill = color_fondo
            ws.cell(row=fila_cabecera, column=col + 3, value="Tiempos de atención por hora").fill = color_fondo
        else:
            columnas_no_encontradas.append("TIEMPO DE SOLUCION REAL")

        # 3. FECHA CIERRE
        col = buscar_columna("FECHACIERRE")
        if col:
            ws.insert_cols(col + 1)
            ws.cell(row=fila_cabecera, column=col + 1, value="Días transcurridos de solución").fill = color_fondo
        else:
            columnas_no_encontradas.append("FECHA CIERRE")

        # 4. DESCRIPCION (Adaptado para 5 columnas: NLP, Clúster 1 y Clúster 2)
        col = buscar_columna("DESCRIPCION")
        if col:
            ws.insert_cols(col + 1, amount=5)
            ws.cell(row=fila_cabecera, column=col + 1, value="Detalle Descripción").fill = color_fondo
            ws.cell(row=fila_cabecera, column=col + 2, value="Score Máximo").fill = color_fondo
            ws.cell(row=fila_cabecera, column=col + 3, value="Debug Scores").fill = color_fondo
            ws.cell(row=fila_cabecera, column=col + 4, value="Nombre del Cluster").fill = color_fondo
            ws.cell(row=fila_cabecera, column=col + 5, value="Nombre del cluster 2").fill = color_fondo
        else:
            columnas_no_encontradas.append("DESCRIPCION")

        # ==========================================
        # FASE 2: BUSCAR UBICACIONES FINALES Y APLICAR FÓRMULAS
        # ==========================================
        
        # Fórmulas de Tiempos
        col_atencion_real = buscar_columna("TIEMPODEATENCIONREAL")
        col_solucion_real = buscar_columna("TIEMPODESOLUCIONREAL")
        col_t_total       = buscar_columna("Tiempos total de atención")
        col_form_hora     = buscar_columna("formula tiempos por hora")
        col_t_hora        = buscar_columna("Tiempos de atención por hora")

        if all([col_atencion_real, col_solucion_real, col_t_total, col_form_hora, col_t_hora]):
            letra_atencion = get_column_letter(col_atencion_real)
            letra_solucion = get_column_letter(col_solucion_real)
            letra_t_total  = get_column_letter(col_t_total)

            for row in range(fila_cabecera + 1, max_row + 1):
                ws.cell(row=row, column=col_t_total, value=f"={letra_atencion}{row}+{letra_solucion}{row}").fill = color_fondo
                ws.cell(row=row, column=col_form_hora, value=f"={letra_t_total}{row}/60").fill = color_fondo
                ws.cell(row=row, column=col_t_hora, value=f"={letra_t_total}{row}/60").fill = color_fondo
        else:
            if not col_atencion_real:
                columnas_no_encontradas.append("TIEMPO DE ATENCION REAL (Necesaria para fórmulas)")

        # --- Fórmulas de Fecha Cierre ---
        col_fecha_cierre = buscar_columna("FECHACIERRE")
        col_fecha_registro = buscar_columna("FECHAREGISTRO")
        col_dias_trans = buscar_columna("Días transcurridos de solución")
        
        if col_fecha_cierre and col_fecha_registro and col_dias_trans:
            letra_fecha_cierre = get_column_letter(col_fecha_cierre)
            letra_fecha_registro = get_column_letter(col_fecha_registro)
            
            for row in range(fila_cabecera + 1, max_row + 1):
                ws.cell(row=row, column=col_dias_trans, value=f"={letra_fecha_cierre}{row}-{letra_fecha_registro}{row}").fill = color_fondo
        else:
            if not col_fecha_registro:
                columnas_no_encontradas.append("FECHA DE REGISTRO (Necesaria para fórmula de días)")

        # --- Procesamiento NLP para Descripciones ---
        col_desc_orig = buscar_columna("DESCRIPCION")
        col_detalle = buscar_columna("Detalle Descripción")
        col_cluster = buscar_columna("Nombre del Cluster")
        col_cluster2 = buscar_columna("Nombre del cluster 2")
        col_score = buscar_columna("Score Máximo")
        col_debug = buscar_columna("Debug Scores")
        
        if col_desc_orig and col_detalle and col_cluster and col_cluster2:
            for row in range(fila_cabecera + 1, max_row + 1):
                texto_query = str(ws.cell(row=row, column=col_desc_orig).value or "")
                
                cat, score, debug_str = motor.clasificar_query(texto_query)
                nombre_cluster = dict_clusters.get(cat, "-")
                nombre_cluster_2 = dict_clusters_2.get(cat, "-")
                
                ws.cell(row=row, column=col_detalle, value=cat).fill = color_fondo
                ws.cell(row=row, column=col_cluster, value=nombre_cluster).fill = color_fondo
                ws.cell(row=row, column=col_cluster2, value=nombre_cluster_2).fill = color_fondo
                ws.cell(row=row, column=col_score, value=score).fill = color_fondo
                ws.cell(row=row, column=col_debug, value=debug_str).fill = color_fondo

        # ==========================================
        # SOLUCIÓN AL ERROR DE REPARACIÓN (TABLAS XML)
        # ==========================================
        if hasattr(ws, 'tables'):
            for table_name in list(ws.tables.keys()):
                del ws.tables[table_name]

        # ==========================================
        # GUARDAR COMO COPIA
        # ==========================================
        directorio, nombre_archivo = os.path.split(ruta_archivo)
        nombre_base, ext = os.path.splitext(nombre_archivo)
        nueva_ruta = os.path.join(directorio, f"{nombre_base}_procesado{ext}")
        
        wb.save(nueva_ruta)

        # ==========================================
        # FASE 3: HOJAS ANALÍTICAS (PANDAS)
        # ==========================================
        try:
            print("Generando tablas dinámicas...")
            df = pd.read_excel(nueva_ruta, header=fila_cabecera-1)
            
            with pd.ExcelWriter(nueva_ruta, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
                
                # Extraer Año de manera independiente para que todas las hojas puedan usarlo.
                # Los casos sin fecha válida quedan como "Sin Año" (no se pierden en los conteos).
                c_fecha_reg = buscar_columna_df(df, 'FECHAREGISTRO', 'FECHA DE REGISTRO')
                if c_fecha_reg:
                    anios = pd.to_datetime(df[c_fecha_reg], errors='coerce', dayfirst=True).dt.year
                    df['Año'] = anios.map(lambda a: 'Sin Año' if pd.isna(a) else str(int(a)))
                else:
                    df['Año'] = "Sin Año"

                # Hoja 1: Casos por Localidad y Año
                if 'SUCURSAL USUARIO' in df.columns:
                    df['Localidad_Limpia'] = df['SUCURSAL USUARIO'].apply(normalizar_localidad)
                    tabla_localidades = pd.crosstab(df['Localidad_Limpia'], df['Año'], margins=True, margins_name="Total")
                    
                    # Orden 01..20, luego localidades raras ("SIN REGISTRO"...), y Total al final
                    tabla_localidades = ordenar_por_localidad(tabla_localidades)
                    
                    tabla_localidades.to_excel(writer, sheet_name='Casos por Localidad')
                else:
                    print("Aviso: No se generó Hoja 1 porque falta 'SUCURSAL USUARIO'.")
                
                # Hoja 2: Conteo de lista específica por año
                if 'Detalle Descripción' in df.columns and lista_conteo_personalizada:
                    def encontrar_mas_cercano(categoria):
                        if pd.isna(categoria) or categoria == "-": return "SIN CLASIFICAR"
                        matches = difflib.get_close_matches(str(categoria).upper(), lista_conteo_personalizada, n=1, cutoff=0.7)
                        return matches[0] if matches else "OTRO"
                        
                    df['Categoria_Lista'] = df['Detalle Descripción'].apply(encontrar_mas_cercano)
                    tabla_categorias = pd.crosstab(df['Categoria_Lista'], df['Año'], margins=True, margins_name="Total")
                    tabla_categorias.to_excel(writer, sheet_name='Conteo Categorías')
                else:
                    print("Aviso: No se generó Hoja 2 porque el TXT de lista estaba vacío o falta la columna descriptiva.")

                # ==========================================
                # Hoja 3: Conteo de Estados POR AÑO
                # + nuevo estado "CERRADO devuelto a la DGDL"
                # ==========================================
                c_estado = buscar_columna_df(df, 'ESTADO')
                if c_estado:
                    estados_validos = ['ANULADO', 'ASIGNADO', 'EN PROCESO', 'SOLUCIONADO', 'CERRADO']
                    df['Estado_Limpio'] = df[c_estado].astype(str).str.strip().str.upper()

                    # Detectar casos devueltos a la DGDL por el comentario de la solución
                    c_comentario = buscar_columna_df(df, 'COMENTARIO DE LA SOLUCION', 'COMENTARIO SOLUCION')
                    if c_comentario:
                        es_devuelto = texto_sin_tildes(df[c_comentario]).str.contains(TEXTO_DGDL, regex=False)
                    else:
                        print("Aviso: No se encontró 'COMENTARIO DE LA SOLUCION'; el conteo DGDL quedará en 0.")
                        es_devuelto = pd.Series(False, index=df.index)
                    print(f"Casos devueltos a la DGDL encontrados: {int(es_devuelto.sum())}")

                    if SEPARAR_DEVUELTOS_DGDL:
                        # Los devueltos pasan a su propio estado (no se cuentan doble)
                        df['Estado_Final'] = df['Estado_Limpio'].where(~es_devuelto, ESTADO_DGDL)
                        df_estados = df[df['Estado_Final'].isin(estados_validos + [ESTADO_DGDL])]
                        tabla_estados = pd.crosstab(df_estados['Estado_Final'], df_estados['Año'])
                    else:
                        # Se mantienen en su estado original y además se agrega la fila DGDL
                        df_estados = df[df['Estado_Limpio'].isin(estados_validos)]
                        tabla_estados = pd.crosstab(df_estados['Estado_Limpio'], df_estados['Año'])
                        fila_dgdl = df[es_devuelto].groupby('Año').size().rename(ESTADO_DGDL)
                        tabla_estados = pd.concat([tabla_estados, fila_dgdl.to_frame().T])

                    # Orden fijo de filas; el estado DGDL aparece siempre (aunque sea 0)
                    orden_estados = estados_validos + [ESTADO_DGDL]
                    tabla_estados = tabla_estados.reindex(orden_estados).fillna(0).astype(int)
                    if tabla_estados.shape[1] == 0:
                        tabla_estados['Sin Año'] = 0
                    tabla_estados['Total'] = tabla_estados.sum(axis=1)
                    if SEPARAR_DEVUELTOS_DGDL:
                        # El total general solo tiene sentido si no hay doble conteo
                        tabla_estados.loc['Total'] = tabla_estados.sum()
                    tabla_estados.index.name = 'Estado'
                    tabla_estados.to_excel(writer, sheet_name='Conteo Estados')
                
                # ----------------------------------------------------------
                # Valores numéricos calculados en pandas.
                # OJO: las columnas creadas con fórmulas (=A2+B2) NO tienen valor
                # cuando pandas lee el archivo, porque openpyxl no calcula fórmulas.
                # Por eso se recalculan aquí desde las columnas originales.
                # ----------------------------------------------------------
                c_at_real = buscar_columna_df(df, 'TIEMPODEATENCIONREAL', 'TIEMPO DE ATENCION REAL')
                c_sol_real = buscar_columna_df(df, 'TIEMPODESOLUCIONREAL', 'TIEMPO DE SOLUCION REAL')
                if c_at_real:
                    df['Atencion_Num'] = a_numero(df[c_at_real])
                if c_sol_real:
                    df['Solucion_Num'] = a_numero(df[c_sol_real])
                if c_at_real and c_sol_real:
                    df['Tiempos_Num'] = df['Atencion_Num'] + df['Solucion_Num']
                elif 'Tiempos total de atención' in df.columns:
                    df['Tiempos_Num'] = a_numero(df['Tiempos total de atención'])

                c_fecha_cierre = buscar_columna_df(df, 'FECHACIERRE', 'FECHA DE CIERRE')
                if c_fecha_cierre and c_fecha_reg:
                    f_cierre = pd.to_datetime(df[c_fecha_cierre], errors='coerce', dayfirst=True)
                    f_reg = pd.to_datetime(df[c_fecha_reg], errors='coerce', dayfirst=True)
                    df['Días_Num'] = (f_cierre - f_reg).dt.total_seconds() / 86400
                elif 'Días transcurridos de solución' in df.columns:
                    df['Días_Num'] = a_numero(df['Días transcurridos de solución'])

                # Preparación de datos de localidad si existen
                if 'SUCURSAL USUARIO' in df.columns:
                    if 'Localidad_Limpia' not in df.columns:
                        df['Localidad_Limpia'] = df['SUCURSAL USUARIO'].apply(normalizar_localidad)
                    
                    # ==========================================
                    # Hojas de PROMEDIOS POR AÑO
                    # (filas = localidad, columnas = años + PROMEDIO GENERAL,
                    #  última fila = PROMEDIO GENERAL de todas las localidades)
                    # ==========================================
                    def hoja_promedio_por_anio(col_valor, nombre_hoja):
                        if col_valor not in df.columns or not df[col_valor].notna().any():
                            print(f"Aviso: No se generó '{nombre_hoja}' porque no hay valores numéricos.")
                            return
                        tabla = pd.pivot_table(
                            df, values=col_valor, index='Localidad_Limpia', columns='Año',
                            aggfunc='mean', margins=True, margins_name='PROMEDIO GENERAL'
                        ).round(2)
                        tabla = ordenar_por_localidad(tabla)
                        tabla.columns.name = 'Año'
                        tabla.index.name = 'Localidad'
                        tabla.to_excel(writer, sheet_name=nombre_hoja)

                    # Hoja 4: Promedio de días transcurridos (cierre - registro)
                    hoja_promedio_por_anio('Días_Num', 'Promedio Días')
                    # Hoja 5: Tiempo total real (atención real + solución real)
                    hoja_promedio_por_anio('Tiempos_Num', 'Prom T Total real')
                    # Hoja 6: Tiempo de atención real
                    hoja_promedio_por_anio('Atencion_Num', 'Prom T Atención R')
                    # Hoja 7: Tiempo de solución real
                    hoja_promedio_por_anio('Solucion_Num', 'Prom T Solución R')
                    
        except Exception as e:
            print(f"Error generando hojas analíticas: {e}")
        
        print(f"\n==========================================")
        if len(columnas_no_encontradas) == 0:
            print("¡Éxito Total! Todas las columnas y fórmulas se aplicaron correctamente.")
        else:
            print("Proceso a Medias. No se encontraron estas columnas:")
            for c in columnas_no_encontradas:
                print(f"- {c}")
        print(f"Ruta: {nueva_ruta}")
        print(f"==========================================\n")

    except Exception as e:
        print(f"Ocurrió un error inesperado:\n{str(e)}")

if __name__ == "__main__":
    procesar_excel()

Paso 1: Seleccionando archivo TXT de categorías...
Paso 1.1: Seleccionando archivo TXT de Clústeres...
Paso 1.2: Seleccionando Excel de mapeo de Clústeres 2...
Paso 1.3: Seleccionando archivo TXT con lista de conteo (Tercera Hoja)...
Paso 2: Seleccionando archivo Excel a procesar...
Procesando Excel...
Generando tablas dinámicas...
Casos devueltos a la DGDL encontrados: 187

¡Éxito Total! Todas las columnas y fórmulas se aplicaron correctamente.
Ruta: C:/Users/Pc/OneDrive/Trabajo/2026/Secretaria de Gobierno/05 Mayo/Reporte de analisis Casos HOLA\fusion_2_archivos_procesado.xlsx



### Para unificar en un solo excel los casos HOLA

In [ ]:
"""
Fusionador de archivos Excel por nombre de columna.

Flujo:
  1. Se abre un explorador para elegir el archivo Excel de REFERENCIA
     (su orden y nombres de columnas mandan).
  2. Se abre otro explorador para elegir UNO o VARIOS archivos a fusionar.
  3. Cada columna de esos archivos se empareja con la columna de referencia
     más parecida (coincidencia exacta normalizada y, si no hay, aproximada).
  4. Se muestra un resumen del emparejamiento para confirmar.
  5. Se guarda un Excel con:
       - Hoja "Datos": todos los registros con las columnas de la referencia.
       - Hoja "Resumen": emparejamientos y filas aportadas por cada archivo.

Requisitos:  pip install pandas openpyxl
"""

import re
import unicodedata
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd
import tkinter as tk
from tkinter import filedialog, messagebox

# Similitud mínima (0 a 1) para aceptar una coincidencia aproximada.
UMBRAL_SIMILITUD = 0.75
# Agregar una columna que indique de qué archivo viene cada fila.
AGREGAR_COLUMNA_ORIGEN = True
NOMBRE_COLUMNA_ORIGEN = "archivo_origen"

TIPOS_EXCEL = [("Archivos Excel", "*.xlsx *.xls *.xlsm"), ("Todos", "*.*")]


# ----------------------------------------------------------------------------
# Utilidades de comparación
# ----------------------------------------------------------------------------
def normalizar(nombre) -> str:
    """Minúsculas, sin tildes, sin signos ni espacios: 'Fecha  de Nacimiento' -> 'fechadenacimiento'."""
    texto = str(nombre).strip().lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]", "", texto)


def similitud(a: str, b: str) -> float:
    return SequenceMatcher(None, normalizar(a), normalizar(b)).ratio()


def emparejar_columnas(cols_ref, cols_otro, umbral=UMBRAL_SIMILITUD):
    """
    Devuelve:
      mapeo      -> {columna_del_otro_archivo: columna_de_referencia}
      detalle    -> lista de dicts para el resumen
      sobrantes  -> columnas del otro archivo sin pareja
      faltantes  -> columnas de referencia sin pareja
    Cada columna de referencia se usa una sola vez.
    """
    mapeo, detalle = {}, []
    ref_libres = list(cols_ref)
    otro_libres = list(cols_otro)

    # 1) Coincidencias exactas tras normalizar
    norm_ref = {normalizar(c): c for c in ref_libres}
    for col in list(otro_libres):
        n = normalizar(col)
        if n in norm_ref and norm_ref[n] in ref_libres:
            destino = norm_ref[n]
            mapeo[col] = destino
            detalle.append({"columna_archivo": col, "columna_referencia": destino,
                            "similitud": 1.0, "tipo": "exacta"})
            ref_libres.remove(destino)
            otro_libres.remove(col)

    # 2) Coincidencias aproximadas: se asignan primero los pares más parecidos
    candidatos = sorted(
        ((similitud(o, r), o, r) for o in otro_libres for r in ref_libres),
        reverse=True,
    )
    for score, o, r in candidatos:
        if score < umbral:
            break
        if o in otro_libres and r in ref_libres:
            mapeo[o] = r
            detalle.append({"columna_archivo": o, "columna_referencia": r,
                            "similitud": round(score, 3), "tipo": "aproximada"})
            otro_libres.remove(o)
            ref_libres.remove(r)

    return mapeo, detalle, otro_libres, ref_libres


# ----------------------------------------------------------------------------
# Lectura
# ----------------------------------------------------------------------------
def leer_excel(ruta: str) -> pd.DataFrame:
    """Lee la primera hoja. Si el archivo tiene varias, avisa cuál se usa."""
    xls = pd.ExcelFile(ruta)
    hoja = xls.sheet_names[0]
    if len(xls.sheet_names) > 1:
        print(f"[Aviso] '{Path(ruta).name}' tiene varias hojas; se usa '{hoja}'.")
    df = pd.read_excel(xls, sheet_name=hoja)
    df.columns = [str(c).strip() for c in df.columns]
    return df


# ----------------------------------------------------------------------------
# Programa principal
# ----------------------------------------------------------------------------
def main():
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    # 1. Archivo de referencia
    ruta_ref = filedialog.askopenfilename(
        title="1) Selecciona el archivo Excel de REFERENCIA", filetypes=TIPOS_EXCEL)
    if not ruta_ref:
        messagebox.showinfo("Cancelado", "No se seleccionó archivo de referencia.")
        return

    # 2. Archivos a fusionar
    rutas_otros = filedialog.askopenfilenames(
        title="2) Selecciona el/los archivo(s) Excel a fusionar", filetypes=TIPOS_EXCEL)
    rutas_otros = [r for r in rutas_otros if Path(r) != Path(ruta_ref)]
    if not rutas_otros:
        messagebox.showinfo("Cancelado", "No se seleccionaron archivos para fusionar.")
        return

    try:
        df_ref = leer_excel(ruta_ref)
    except Exception as e:
        messagebox.showerror("Error", f"No se pudo leer la referencia:\n{e}")
        return

    cols_ref = list(df_ref.columns)
    nombre_ref = Path(ruta_ref).name
    if AGREGAR_COLUMNA_ORIGEN:
        df_ref[NOMBRE_COLUMNA_ORIGEN] = nombre_ref

    partes = [df_ref]
    resumen_filas = [{"archivo": nombre_ref, "filas": len(df_ref),
                      "columna_archivo": "(referencia)", "columna_referencia": "",
                      "similitud": "", "tipo": ""}]
    avisos = []

    # 3. Emparejar y reordenar cada archivo
    for ruta in rutas_otros:
        nombre = Path(ruta).name
        try:
            df = leer_excel(ruta)
        except Exception as e:
            avisos.append(f"✖ {nombre}: no se pudo leer ({e}). Se omite.")
            continue

        mapeo, detalle, sobrantes, faltantes = emparejar_columnas(cols_ref, list(df.columns))

        df = df.rename(columns=mapeo)
        df = df.reindex(columns=cols_ref)  # orden de la referencia; faltantes quedan vacías
        if AGREGAR_COLUMNA_ORIGEN:
            df[NOMBRE_COLUMNA_ORIGEN] = nombre
        partes.append(df)

        for d in detalle:
            resumen_filas.append({"archivo": nombre, "filas": len(df), **d})
        for s in sobrantes:
            resumen_filas.append({"archivo": nombre, "filas": len(df), "columna_archivo": s,
                                  "columna_referencia": "(SIN PAREJA - descartada)",
                                  "similitud": "", "tipo": "sobrante"})
        for f in faltantes:
            resumen_filas.append({"archivo": nombre, "filas": len(df), "columna_archivo": "(no existe)",
                                  "columna_referencia": f, "similitud": "", "tipo": "faltante"})

        aprox = [d for d in detalle if d["tipo"] == "aproximada"]
        linea = f"• {nombre}: {len(detalle)}/{len(cols_ref)} columnas emparejadas"
        if aprox:
            linea += "\n    Aproximadas: " + "; ".join(
                f"'{d['columna_archivo']}' → '{d['columna_referencia']}' ({d['similitud']:.0%})"
                for d in aprox)
        if sobrantes:
            linea += f"\n    Sin pareja (se descartan): {', '.join(sobrantes)}"
        if faltantes:
            linea += f"\n    Quedarán vacías: {', '.join(faltantes)}"
        avisos.append(linea)

    if len(partes) < 2:
        messagebox.showerror("Error", "No hubo archivos válidos para fusionar.\n\n" + "\n".join(avisos))
        return

    # 4. Confirmación
    total_archivos = len(partes)
    msg = (f"Referencia: {nombre_ref}\n"
           f"Archivos a fusionar (incluida la referencia): {total_archivos}\n\n"
           + "\n".join(avisos)
           + "\n\n¿Deseas continuar y guardar el resultado?")
    if not messagebox.askyesno("Resumen del emparejamiento", msg):
        return

    resultado = pd.concat(partes, ignore_index=True)

    # 5. Guardar
    ruta_salida = filedialog.asksaveasfilename(
        title="Guardar archivo fusionado",
        defaultextension=".xlsx",
        initialdir=str(Path(ruta_ref).parent),
        initialfile=f"fusion_{total_archivos}_archivos.xlsx",
        filetypes=[("Archivo Excel", "*.xlsx")],
    )
    if not ruta_salida:
        return

    try:
        with pd.ExcelWriter(ruta_salida, engine="openpyxl") as writer:
            resultado.to_excel(writer, sheet_name="Datos", index=False)
            pd.DataFrame(resumen_filas).to_excel(writer, sheet_name="Resumen", index=False)
    except PermissionError:
        messagebox.showerror("Error", "No se pudo guardar. ¿El archivo está abierto en Excel?")
        return

    messagebox.showinfo(
        "Listo",
        f"Se fusionaron {total_archivos} archivos ({len(resultado):,} filas).\n\nGuardado en:\n{ruta_salida}")


if __name__ == "__main__":
    main()